# 기존

In [ ]:
from pathlib import Path
from typing import Any, Dict, List, Optional, Tuple
import re
import json

# 여기에 "본인 값" 입력
DB_USER = "dbuser"
DB_PASS = "jnudl1"           # 실제 비번
DB_HOST = "168.131.30.66"
DB_PORT = "5432"
DB_NAME = "coconut_240508"

HF_TOKEN = "REMOVED"  # 본인 HF 토큰
MODEL_ID = "google/gemma-3-12b-it"            # GPU 여유되면 2b-it로 올려도 됨
CUDA_VISIBLE = "0"                              # 3번 GPU 쓰고 싶을 때

env_text = f"""# ---- runtime secrets ----
DATABASE_URL=postgresql://{DB_USER}:{DB_PASS}@{DB_HOST}:{DB_PORT}/{DB_NAME}
HF_TOKEN={HF_TOKEN}
MODEL_ID={MODEL_ID}
CUDA_VISIBLE_DEVICES={CUDA_VISIBLE}
"""

Path(".env").write_text(env_text)
print("Wrote .env at:", Path(".env").resolve())

In [ ]:
import os, torch
from huggingface_hub import login
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

# HF 로그인
login(os.environ["HF_TOKEN"])

# 노트북에서 환경변수 적용(안 되어있으면 강제 세팅)
os.environ["CUDA_VISIBLE_DEVICES"] = os.getenv("CUDA_VISIBLE_DEVICES", "0")

use_4bit = True
quant = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16
) if use_4bit else None

MODEL_ID = os.getenv("MODEL_ID", "google/gemma-3-12b-it")
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, use_fast=False)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.bfloat16 if torch.cuda.is_available() else torch.float32,
    quantization_config=quant,
    device_map="auto",
)
print("Loaded:", MODEL_ID)


# 신규

In [2]:
"""
Clinical Trial Protocol Generator using LLaMA 3.1 with Few-shot CoT
- Hugging Face에서 LLaMA 3.1 Instruct 모델 사용
- Few-shot Chain of Thought 방식으로 프로토콜 생성
- Study Overview, Participation Criteria, Study Plan 생성
"""

import os
import json
from dotenv import load_dotenv
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline

# ============================================================
# 1. 환경 설정 및 모델 로딩
# ============================================================

# .env 파일에서 HF_TOKEN 로드
load_dotenv()
HF_TOKEN = os.getenv("HF_TOKEN")

# 모델 설정
MODEL_NAME = "meta-llama/Llama-3.1-8B-Instruct"  # 또는 70B 버전

def load_model():
    """LLaMA 3.1 모델 로딩"""
    print("🔄 모델 로딩 중...")
    
    tokenizer = AutoTokenizer.from_pretrained(
        MODEL_NAME,
        token=HF_TOKEN,
        trust_remote_code=True
    )
    
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME,
        token=HF_TOKEN,
        torch_dtype=torch.float16,  # 메모리 절약
        device_map="auto",          # GPU 자동 할당
        trust_remote_code=True
    )
    
    # 파이프라인 생성
    pipe = pipeline(
        "text-generation",
        model=model,
        tokenizer=tokenizer,
        torch_dtype=torch.float16,
        device_map="auto"
    )
    
    print("✅ 모델 로딩 완료!")
    return pipe, tokenizer


# ============================================================
# 2. Few-shot 예시 정의 (이 부분을 수정하세요!)
# ============================================================

FEW_SHOT_EXAMPLES = [
    # ========== 예시 1 ==========
    {
        "input": """
사용자 입력 정보:
- 연구 주제: 제2형 당뇨병 환자에서 신약 A의 효과
- 연구 목적: 혈당 조절 개선
- 대상: 성인 당뇨병 환자
- 기간: 24주
""",
        "reasoning": """
추론 과정:
1. Study Overview 분석:
   - 연구 주제가 "제2형 당뇨병 신약"이므로 Study Type은 INTERVENTIONAL
   - 신약 연구이므로 Phase는 PHASE3가 적절
   - 24주 연구이므로 Primary Completion Date 설정 필요

2. Participation Criteria 분석:
   - "성인 당뇨병 환자"이므로 Minimum Age는 18 Years
   - 당뇨병 환자 대상이므로 Healthy Volunteers는 No
   - 성별 제한 없음 → Sex: ALL

3. Study Plan 분석:
   - 신약 효과 연구이므로 Primary Purpose: TREATMENT
   - 위약 대조 필요 → Allocation: RANDOMIZED
   - 이중맹검이 표준 → Masking: DOUBLE
""",
        "output": {
            "Study Overview": {
                "NCT ID": "NCT00000001",
                "Brief Title": "Study of Drug A in Type 2 Diabetes",
                "Official Title": "A Phase 3, Randomized, Double-Blind Study of Drug A for Glycemic Control in Patients with Type 2 Diabetes",
                "Brief Summary": "This study evaluates the efficacy and safety of Drug A compared to placebo in improving glycemic control in adults with Type 2 Diabetes Mellitus over 24 weeks.",
                "Conditions": ["Type 2 Diabetes Mellitus"],
                "Study Type": "INTERVENTIONAL",
                "Phases": ["PHASE3"],
                "Enrollment": 300,
                "Overall Status": "RECRUITING",
                "Lead Sponsor": "Pharmaceutical Company X",
                "Interventions": [
                    {
                        "Type": "DRUG",
                        "Name": "Drug A",
                        "Description": "Oral administration of Drug A 10mg once daily for 24 weeks"
                    }
                ]
            },
            "Participation Criteria": {
                "Eligibility Criteria": "Inclusion Criteria:\\n* Adults aged 18-75 years\\n* Diagnosed with Type 2 Diabetes Mellitus\\n* HbA1c between 7.0% and 10.0%\\n\\nExclusion Criteria:\\n* Type 1 Diabetes\\n* Pregnant or breastfeeding women\\n* Severe renal impairment (eGFR < 30)",
                "Healthy Volunteers": "No",
                "Sex": "ALL",
                "Minimum Age": "18 Years",
                "Maximum Age": "75 Years",
                "Standard Ages": ["ADULT", "OLDER_ADULT"]
            },
            "Study Plan": {
                "Design Information": {
                    "Allocation": "RANDOMIZED",
                    "Intervention Model": "PARALLEL",
                    "Primary Purpose": "TREATMENT",
                    "Masking": "DOUBLE",
                    "Who Masked": ["PARTICIPANT", "INVESTIGATOR"]
                },
                "Arm Groups": [
                    {
                        "Label": "Drug A Group",
                        "Type": "EXPERIMENTAL",
                        "Description": "Participants receive Drug A 10mg orally once daily"
                    },
                    {
                        "Label": "Placebo Group",
                        "Type": "PLACEBO_COMPARATOR",
                        "Description": "Participants receive matching placebo orally once daily"
                    }
                ],
                "Primary Outcomes": [
                    {
                        "Measure": "Change in HbA1c from baseline",
                        "Time Frame": "24 weeks"
                    }
                ],
                "Secondary Outcomes": [
                    {
                        "Measure": "Change in fasting plasma glucose",
                        "Time Frame": "24 weeks"
                    }
                ]
            }
        }
    },
    
    # ========== 예시 2 ==========
    {
        "input": """
사용자 입력 정보:
- 연구 주제: 우울증 환자에서 인지행동치료의 효과
- 연구 목적: 우울 증상 개선
- 대상: 주요우울장애 진단 성인
- 기간: 12주
""",
        "reasoning": """
추론 과정:
1. Study Overview 분석:
   - 인지행동치료는 행동적 중재이므로 Intervention Type은 BEHAVIORAL
   - 치료 연구이므로 Study Type은 INTERVENTIONAL
   - 비약물 연구이므로 Phase는 NA

2. Participation Criteria 분석:
   - "성인" 대상이므로 Minimum Age: 18 Years
   - 우울증 환자 대상이므로 Healthy Volunteers: No
   - 정신건강 연구에서 자살 위험 제외 필요

3. Study Plan 분석:
   - 치료 효과 연구 → Primary Purpose: TREATMENT
   - 행동치료는 블라인딩 어려움 → Masking: NONE (Open Label)
   - 대기자 명단 대조군 사용 가능
""",
        "output": {
            "Study Overview": {
                "NCT ID": "NCT00000002",
                "Brief Title": "Cognitive Behavioral Therapy for Depression",
                "Official Title": "A Randomized Controlled Trial of Cognitive Behavioral Therapy for Major Depressive Disorder",
                "Brief Summary": "This study evaluates the effectiveness of cognitive behavioral therapy (CBT) compared to treatment as usual in adults with major depressive disorder over 12 weeks.",
                "Conditions": ["Major Depressive Disorder", "Depression"],
                "Study Type": "INTERVENTIONAL",
                "Phases": ["NA"],
                "Enrollment": 150,
                "Overall Status": "RECRUITING",
                "Lead Sponsor": "University Medical Center",
                "Interventions": [
                    {
                        "Type": "BEHAVIORAL",
                        "Name": "Cognitive Behavioral Therapy",
                        "Description": "12 weekly sessions of individual CBT, each lasting 50 minutes"
                    }
                ]
            },
            "Participation Criteria": {
                "Eligibility Criteria": "Inclusion Criteria:\\n* Adults aged 18-65 years\\n* Diagnosis of Major Depressive Disorder (DSM-5)\\n* PHQ-9 score >= 10\\n\\nExclusion Criteria:\\n* Active suicidal ideation\\n* Current psychotic symptoms\\n* Substance use disorder in past 6 months",
                "Healthy Volunteers": "No",
                "Sex": "ALL",
                "Minimum Age": "18 Years",
                "Maximum Age": "65 Years",
                "Standard Ages": ["ADULT", "OLDER_ADULT"]
            },
            "Study Plan": {
                "Design Information": {
                    "Allocation": "RANDOMIZED",
                    "Intervention Model": "PARALLEL",
                    "Primary Purpose": "TREATMENT",
                    "Masking": "NONE",
                    "Who Masked": []
                },
                "Arm Groups": [
                    {
                        "Label": "CBT Group",
                        "Type": "EXPERIMENTAL",
                        "Description": "Participants receive 12 weeks of cognitive behavioral therapy"
                    },
                    {
                        "Label": "Treatment as Usual",
                        "Type": "ACTIVE_COMPARATOR",
                        "Description": "Participants continue with their usual care"
                    }
                ],
                "Primary Outcomes": [
                    {
                        "Measure": "Change in PHQ-9 depression score",
                        "Time Frame": "12 weeks"
                    }
                ],
                "Secondary Outcomes": [
                    {
                        "Measure": "Change in quality of life (SF-36)",
                        "Time Frame": "12 weeks"
                    }
                ]
            }
        }
    },
    
    # ========== 예시 3 (필요시 추가) ==========
    # 진단 연구, 관찰 연구 등 다른 유형 추가 가능
]


# ============================================================
# 3. 프롬프트 템플릿 (이 부분을 수정하세요!)
# ============================================================

SYSTEM_PROMPT = """당신은 임상시험 프로토콜 전문가입니다. 
사용자가 제공하는 연구 정보를 바탕으로 완전한 임상시험 프로토콜을 생성합니다.

생성해야 할 섹션:
1. Study Overview: 연구의 기본 정보 (제목, 요약, 조건, 유형 등)
2. Participation Criteria: 참여 기준 (포함/제외 기준, 연령, 성별 등)
3. Study Plan: 연구 설계 (배정, 눈가림, 결과 측정 등)

반드시 JSON 형식으로 출력하세요."""


def build_few_shot_prompt(user_input: str) -> str:
    """Few-shot CoT 프롬프트 구성"""
    
    prompt = f"<|begin_of_text|><|start_header_id|>system<|end_header_id|>\n\n{SYSTEM_PROMPT}<|eot_id|>"
    
    # Few-shot 예시 추가
    for i, example in enumerate(FEW_SHOT_EXAMPLES, 1):
        prompt += f"""
<|start_header_id|>user<|end_header_id|>

{example['input']}<|eot_id|>

<|start_header_id|>assistant<|end_header_id|>

{example['reasoning']}

최종 출력:
```json
{json.dumps(example['output'], ensure_ascii=False, indent=2)}
```<|eot_id|>
"""
    
    # 실제 사용자 입력 추가
    prompt += f"""
<|start_header_id|>user<|end_header_id|>

{user_input}<|eot_id|>

<|start_header_id|>assistant<|end_header_id|>

"""
    
    return prompt


# ============================================================
# 4. 생성 함수
# ============================================================

def generate_protocol(pipe, user_input: str, max_new_tokens: int = 2048) -> dict:
    """임상시험 프로토콜 생성"""
    
    # Few-shot 프롬프트 구성
    prompt = build_few_shot_prompt(user_input)
    
    # 생성
    outputs = pipe(
        prompt,
        max_new_tokens=max_new_tokens,
        do_sample=True,
        temperature=0.7,
        top_p=0.9,
        repetition_penalty=1.1,
        return_full_text=False
    )
    
    generated_text = outputs[0]["generated_text"]
    
    # JSON 추출
    try:
        json_start = generated_text.find("```json")
        json_end = generated_text.find("```", json_start + 7)
        
        if json_start != -1 and json_end != -1:
            json_str = generated_text[json_start + 7:json_end].strip()
            result = json.loads(json_str)
        else:
            # JSON 블록이 없으면 전체에서 추출 시도
            json_start = generated_text.find("{")
            json_end = generated_text.rfind("}") + 1
            json_str = generated_text[json_start:json_end]
            result = json.loads(json_str)
            
    except json.JSONDecodeError:
        print("⚠️ JSON 파싱 실패. 원본 텍스트 반환.")
        result = {"raw_output": generated_text}
    
    return result, generated_text


# ============================================================
# 5. 메인 실행
# ============================================================

def main():
    # 모델 로딩
    pipe, tokenizer = load_model()
    print("로딩완료")
    # 사용자 입력 예시
    user_input = """
사용자 입력 정보:
- 연구 주제: 폐색전증 의심 환자에서 초음파 검사의 진단적 유용성
- 연구 목적: CT 또는 폐환기관류스캔 의뢰 감소
- 대상: 응급실 내원 성인 환자
- 중재: Point-of-care 초음파 검사
- 기간: 24시간 내 결과 확인
"""
    
    print("\n" + "="*70)
    print("📋 임상시험 프로토콜 생성")
    print("="*70)
    print(f"\n입력:\n{user_input}")
    
    # 프로토콜 생성
    result, raw_output = generate_protocol(pipe, user_input)
    
    print("\n" + "="*70)
    print("✅ 생성 결과")
    print("="*70)
    print(json.dumps(result, ensure_ascii=False, indent=2))
    
    # 결과 저장
    with open("Results/generated_protocol.json", "w", encoding="utf-8") as f:
        json.dump(result, f, ensure_ascii=False, indent=2)
    print("\n💾 결과가 'Results/generated_protocol.json'에 저장되었습니다.")
    
    return result


if __name__ == "__main__":
    main()


🔄 모델 로딩 중...


Loading checkpoint shards: 100%|██████████| 4/4 [00:01<00:00,  2.78it/s]
Device set to use cpu
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


✅ 모델 로딩 완료!
로딩완료

📋 임상시험 프로토콜 생성

입력:

사용자 입력 정보:
- 연구 주제: 폐색전증 의심 환자에서 초음파 검사의 진단적 유용성
- 연구 목적: CT 또는 폐환기관류스캔 의뢰 감소
- 대상: 응급실 내원 성인 환자
- 중재: Point-of-care 초음파 검사
- 기간: 24시간 내 결과 확인


✅ 생성 결과
{
  "Study Overview": {
    "NCT ID": "NCT00000003",
    "Brief Title": "Diagnostic Utility of Point-of-Care Ultrasound in Suspected Pneumothorax",
    "Official Title": "A Prospective Cohort Study Evaluating the Diagnostic Accuracy of Point-of-Care Ultrasound in Emergency Department Patients with Suspected Pneumothorax",
    "Brief Summary": "This study assesses the diagnostic utility of point-of-care ultrasound in emergency department patients with suspected pneumothorax.",
    "Conditions": [
      "Pneumothorax"
    ],
    "Study Type": "DIAGNOSTIC",
    "Phases": [
      "NA"
    ],
    "Enrollment": 200,
    "Overall Status": "RECRUITING",
    "Lead Sponsor": "Academic Medical Center",
    "Interventions": [
      {
        "Type": "DEVICE",
        "Name": "Point-of-Care Ultrasound Device",